# IBKR Flex sync

Pulls the "Trade History API" Flex Query (Cash Report + Open Positions + Trades) and brings `data/brokers/ibkr/ledger.csv` up to date. Safe to re-run: ledger events dedupe by `event_id`, so running this twice in a row just confirms nothing changed.

Run this regularly — the underlying Flex Query is scoped to "Last Business Day" on IBKR's side, so a missed run creates a permanent gap rather than something you can ask for later. If a run raises `TradeHistoryGapError`, see `docs/ibkr_flex_api.md` for how to backfill it. All the mechanics (protocol, error codes, the ledger vs. the raw archive) are documented there too.

In [1]:
from trades.brokers.ibkr import api, main, preprocessing
from trades.config import IbkrFlexApiConfig, IbkrFlexCredentials

credentials = IbkrFlexCredentials()  # reads IBKR_FLEX_WEB_SERVICE_TOKEN / IBKR_QUERY_ID from .env
config = IbkrFlexApiConfig()

## Run the sync

One network round trip (SendRequest, then poll GetStatement until ready), then the ledger is updated from that single fetched statement's `<Trade>` rows.

In [2]:
result = main.sync_ibkr_account(credentials, config)
result

IbkrSyncResult(pulled_at=datetime.datetime(2026, 7, 3, 4, 2, 57), statement_from_date=datetime.date(2025, 7, 3), statement_to_date=datetime.date(2026, 7, 2), new_event_count=0, total_event_count=75)

## What's in the cache now

In [3]:
ledger = main.load_ledger(config)
first_date = ledger["event_datetime"].min().date()
last_date = ledger["event_datetime"].max().date()
print(f"{len(ledger)} ledger events cached, spanning {first_date} to {last_date}")
ledger

75 ledger events cached, spanning 2026-01-25 to 2026-07-01


,event_id,event_datetime,symbol,event_type,shares,price,amount,currency,meta
0,ibkr:37540824085,2026-01-25 23:06:15,CASH,DEPOSIT,NaN,NaN,100.000000,USD,"{'transaction_id': '37540824085', 'type': 'Dep..."
1,ibkr:37591258188,2026-01-27 14:32:23,VOO,BUY,0.1500,640.39,96.058500,USD,"{'transaction_id': '37591258188', 'trade_id': ..."
2,ibkr:37591258188:fee,2026-01-27 14:32:23,VOO,FEE,NaN,NaN,0.960585,USD,"{'transaction_id': '37591258188', 'trade_id': ..."
3,ibkr:38962164586,2026-03-31 20:20:00,VOO,DIVIDEND,NaN,NaN,0.280000,USD,"{'transaction_id': '38962164586', 'type': 'Div..."
4,ibkr:38962164609,2026-03-31 20:20:00,VOO,WITHHOLDING,NaN,NaN,0.040000,USD,"{'transaction_id': '38962164609', 'type': 'Wit..."
...,...,...,...,...,...,...,...,...,...
70,ibkr:41072945401,2026-06-30 20:20:00,VOO,DIVIDEND,NaN,NaN,26.770000,USD,"{'transaction_id': '41072945401', 'type': 'Div..."
71,ibkr:41072945411,2026-06-30 20:20:00,VOO,WITHHOLDING,NaN,NaN,8.030000,USD,"{'transaction_id': '41072945411', 'type': 'Wit..."
72,ibkr:41085895127,2026-07-01 00:00:00,CASH,DEPOSIT,NaN,NaN,2500.000000,USD,"{'transaction_id': '41085895127', 'type': 'Dep..."
73,ibkr:41092706716,2026-07-01 09:30:00,VOO,BUY,0.0273,684.60,18.689580,USD,"{'transaction_id': '41092706716', 'trade_id': ..."
